# First-Order Optimization Methods
## From Gradient Descent to Adam — Theory, Implementation, and Comparison

---

**Author:** Computational Mathematics Notebook Series  
**Topic:** Gradient Descent, Momentum, Adaptive Methods, SGD  
**Prerequisites:** Multivariate calculus, linear algebra, basic convex analysis  
**Primary Reference:** Nocedal, J., & Wright, S. J. (2006). *Numerical Optimization*. Springer.

In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from mpl_toolkits.mplot3d import Axes3D
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.figsize': (12, 5),
    'font.size': 12,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'lines.linewidth': 2
})

print("All imports successful.")

In [ ]:
# =============================================================================
# Global constants and configuration
# =============================================================================

SEED      = 42
TOL       = 1e-8          # convergence tolerance
MAX_ITER  = 2000          # maximum iterations

# Default learning rates per optimizer
LR_GD       = 0.001
LR_MOMENTUM = 0.001
LR_NESTEROV = 0.001
LR_ADAGRAD  = 0.1
LR_RMSPROP  = 0.01
LR_ADAM     = 0.01
LR_AMSGRAD  = 0.01
LR_SGD      = 0.005

# Colour palette — consistent across all plots
COLORS = {
    'gd':       '#1f77b4',   # blue
    'momentum': '#ff7f0e',   # orange
    'nesterov': '#2ca02c',   # green
    'adagrad':  '#d62728',   # red
    'rmsprop':  '#9467bd',   # purple
    'adam':     '#8c564b',   # brown
    'amsgrad':  '#e377c2',   # pink
    'sgd':      '#7f7f7f',   # grey
}

rng = np.random.default_rng(SEED)
print(f"SEED={SEED}, TOL={TOL}, MAX_ITER={MAX_ITER}")

# =============================================================================
# 1. Problem Statement
# =============================================================================

## 1.1 The Unconstrained Minimisation Problem

We seek to solve

$$\min_{x \in \mathbb{R}^n} f(x)$$

where $f : \mathbb{R}^n \to \mathbb{R}$ is differentiable (and usually, though not always, convex).  
We assume access only to a **first-order oracle**: given $x$, it returns the function value $f(x)$ and the gradient $\nabla f(x)$.

## 1.2 Why First-Order Methods?

| Method class | Information used | Cost per step | Typical scale |
|---|---|---|---|
| Zero-order | $f(x)$ only | $O(n)$ evaluations | $n \leq 10^3$ |
| **First-order** | $f(x),\, \nabla f(x)$ | **$O(n)$** | **$n \leq 10^9$** |
| Second-order | $f, \nabla f, \nabla^2 f$ | $O(n^2)$–$O(n^3)$ | $n \leq 10^4$ |

First-order methods dominate machine learning because:
- Gradients are cheap via automatic differentiation (backpropagation).
- Memory is $O(n)$ — feasible for billions of parameters.
- Mini-batch stochastic variants generalise better than exact second-order methods.

## 1.3 Notation

| Symbol | Meaning |
|---|---|
| $x_k$ | iterate at step $k$ |
| $\alpha_k$ or $\eta$ | learning rate (step size) |
| $x^*$ | a global (or local) minimiser |
| $f^* = f(x^*)$ | optimal value |
| $\nabla f(x)$ | gradient (column vector) |
| $L$ | Lipschitz constant of $\nabla f$ |
| $\mu$ | strong convexity constant |
| $\kappa = L/\mu$ | condition number |

# =============================================================================
# 2. Convergence Theory Foundations
# =============================================================================

## 2.1 L-Smoothness (Lipschitz Gradient)

**Definition.** $f$ is *$L$-smooth* if its gradient is Lipschitz continuous:

$$\|\nabla f(x) - \nabla f(y)\| \leq L\,\|x - y\| \quad \forall x, y \in \mathbb{R}^n.$$

Equivalently (for $C^2$ functions): $\nabla^2 f(x) \preceq L\,I$ for all $x$.

## 2.2 The Descent Lemma (Proof)

**Lemma.** If $f$ is $L$-smooth, then for all $x, y$:

$$\boxed{f(y) \leq f(x) + \langle \nabla f(x),\, y - x \rangle + \frac{L}{2}\|y - x\|^2}$$

*Proof sketch.* By the fundamental theorem of calculus:
$$f(y) - f(x) = \int_0^1 \langle \nabla f(x + t(y-x)),\, y-x \rangle \, dt.$$
Subtracting $\langle \nabla f(x), y-x\rangle$ and applying Cauchy–Schwarz plus L-smoothness:
$$|f(y) - f(x) - \langle \nabla f(x), y-x \rangle| \leq \int_0^1 t\,L\,\|y-x\|^2\,dt = \frac{L}{2}\|y-x\|^2. \quad \square$$

## 2.3 Strong Convexity

**Definition.** $f$ is *$\mu$-strongly convex* ($\mu > 0$) if:

$$f(y) \geq f(x) + \langle \nabla f(x),\, y - x\rangle + \frac{\mu}{2}\|y - x\|^2 \quad \forall x, y.$$

Equivalently: $\nabla^2 f(x) \succeq \mu\,I$.

Strong convexity implies a unique minimiser $x^*$ and that function values control the distance:
$$\|x - x^*\|^2 \leq \frac{2}{\mu}(f(x) - f^*).$$

## 2.4 Condition Number

When $\mu \leq L$, the **condition number** is:

$$\kappa = \frac{L}{\mu} \geq 1.$$

Gradient descent converges at rate $(1 - 1/\kappa)^k$. Large $\kappa$ means slow convergence — this is why ill-conditioned problems (e.g., the Rosenbrock banana) are hard for plain GD.

## 2.5 Summary of Convergence Rates

| Setting | GD rate | Accelerated rate |
|---|---|---|
| Convex, $L$-smooth | $O(1/k)$ | $O(1/k^2)$ |
| $\mu$-strongly convex | $O((1-1/\kappa)^k)$ | $O((1-1/\sqrt{\kappa})^k)$ |

# =============================================================================
# 3. Test Functions
# =============================================================================

We use three canonical benchmark functions throughout the notebook.

**Rosenbrock** ($f^* = 0$ at $x^* = (1,1)$, highly ill-conditioned banana valley):
$$f(x_1, x_2) = 100(x_2 - x_1^2)^2 + (1 - x_1)^2$$

**Beale** ($f^* = 0$ at $x^* = (3, 0.5)$):
$$f(x_1, x_2) = (1.5 - x_1 + x_1 x_2)^2 + (2.25 - x_1 + x_1 x_2^2)^2 + (2.625 - x_1 + x_1 x_2^3)^2$$

**Quadratic** (controlled condition number):
$$f(x) = \tfrac{1}{2} x^\top Q x, \quad Q \succ 0$$

In [ ]:
# =============================================================================
# Test function definitions — value + analytical gradient
# =============================================================================

def rosenbrock(x):
    """Rosenbrock (banana) function and gradient.

    Args:
        x: array of shape (2,)

    Returns:
        f:  scalar function value
        g:  gradient, shape (2,)
    """
    x1, x2 = x[0], x[1]
    f = 100.0 * (x2 - x1**2)**2 + (1.0 - x1)**2
    g = np.array([
        -400.0 * x1 * (x2 - x1**2) - 2.0 * (1.0 - x1),
         200.0 * (x2 - x1**2)
    ])
    return f, g


def beale(x):
    """Beale function and gradient.

    Args:
        x: array of shape (2,)

    Returns:
        f:  scalar function value
        g:  gradient, shape (2,)
    """
    x1, x2 = x[0], x[1]
    t1 = 1.5   - x1 + x1 * x2
    t2 = 2.25  - x1 + x1 * x2**2
    t3 = 2.625 - x1 + x1 * x2**3
    f  = t1**2 + t2**2 + t3**2

    dt1_dx1 = -1.0 + x2
    dt2_dx1 = -1.0 + x2**2
    dt3_dx1 = -1.0 + x2**3
    dt1_dx2 =  x1
    dt2_dx2 =  2.0 * x1 * x2
    dt3_dx2 =  3.0 * x1 * x2**2

    g = np.array([
        2.0 * (t1 * dt1_dx1 + t2 * dt2_dx1 + t3 * dt3_dx1),
        2.0 * (t1 * dt1_dx2 + t2 * dt2_dx2 + t3 * dt3_dx2)
    ])
    return f, g


def quadratic(x, Q):
    """Quadratic 0.5 x^T Q x with gradient Q x.

    Args:
        x: array of shape (n,)
        Q: positive definite matrix of shape (n, n)

    Returns:
        f:  scalar function value
        g:  gradient, shape (n,)
    """
    g = Q @ x
    f = 0.5 * x @ g
    return f, g


# Quick sanity check
x_test = np.array([1.0, 1.0])
f_rb, g_rb = rosenbrock(x_test)
print(f"Rosenbrock at optimum (1,1): f = {f_rb:.6f}, ||g|| = {np.linalg.norm(g_rb):.6f}")

x_test2 = np.array([3.0, 0.5])
f_be, g_be = beale(x_test2)
print(f"Beale at optimum   (3,0.5): f = {f_be:.2e}, ||g|| = {np.linalg.norm(g_be):.2e}")

In [ ]:
# =============================================================================
# Visualise test functions
# =============================================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- Rosenbrock ---
ax = axes[0]
x1r = np.linspace(-2, 2, 400)
x2r = np.linspace(-1, 3, 400)
X1r, X2r = np.meshgrid(x1r, x2r)
Fr = 100*(X2r - X1r**2)**2 + (1 - X1r)**2
cs = ax.contourf(X1r, X2r, np.log1p(Fr), levels=50, cmap='viridis')
ax.contour(X1r, X2r, np.log1p(Fr), levels=20, colors='white', alpha=0.3, linewidths=0.5)
ax.plot(1, 1, 'r*', markersize=14, label='$x^*=(1,1)$')
plt.colorbar(cs, ax=ax, label='$\\log(1+f)$')
ax.set_title('Rosenbrock Function')
ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
ax.legend()

# --- Beale ---
ax = axes[1]
x1b = np.linspace(-4.5, 4.5, 400)
x2b = np.linspace(-4.5, 4.5, 400)
X1b, X2b = np.meshgrid(x1b, x2b)
T1 = 1.5   - X1b + X1b*X2b
T2 = 2.25  - X1b + X1b*X2b**2
T3 = 2.625 - X1b + X1b*X2b**3
Fb = T1**2 + T2**2 + T3**2
cs2 = ax.contourf(X1b, X2b, np.log1p(Fb), levels=50, cmap='plasma')
ax.contour(X1b, X2b, np.log1p(Fb), levels=20, colors='white', alpha=0.3, linewidths=0.5)
ax.plot(3, 0.5, 'r*', markersize=14, label='$x^*=(3,0.5)$')
plt.colorbar(cs2, ax=ax, label='$\\log(1+f)$')
ax.set_title('Beale Function')
ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
ax.legend()

# --- Quadratic with kappa=10 ---
ax = axes[2]
Q10 = np.diag([1.0, 10.0])
x1q = np.linspace(-3, 3, 300)
x2q = np.linspace(-3, 3, 300)
X1q, X2q = np.meshgrid(x1q, x2q)
Fq = 0.5*(Q10[0,0]*X1q**2 + Q10[1,1]*X2q**2)
cs3 = ax.contourf(X1q, X2q, Fq, levels=30, cmap='coolwarm')
ax.contour(X1q, X2q, Fq, levels=15, colors='black', alpha=0.4, linewidths=0.5)
ax.plot(0, 0, 'r*', markersize=14, label='$x^*=(0,0)$')
plt.colorbar(cs3, ax=ax, label='$f(x)$')
ax.set_title('Quadratic ($\\kappa=10$)')
ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
ax.legend()

plt.suptitle('Test Functions', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

# =============================================================================
# 4. Vanilla Gradient Descent
# =============================================================================

## 4.1 The Update Rule

The simplest descent method follows the negative gradient:

$$\boxed{x_{k+1} = x_k - \alpha_k \nabla f(x_k)}$$

With a **fixed** step size $\alpha = 1/L$, the descent lemma gives:
$$f(x_{k+1}) \leq f(x_k) - \frac{1}{2L}\|\nabla f(x_k)\|^2.$$

## 4.2 Convergence Rates

**Convex, $L$-smooth** (telescoping the descent inequality):
$$\boxed{f(x_k) - f^* \leq \frac{L\,\|x_0 - x^*\|^2}{2k} = O\!\left(\frac{1}{k}\right)}$$

**$\mu$-strongly convex, $L$-smooth** (geometric series):
$$\boxed{f(x_k) - f^* \leq \left(1 - \frac{\mu}{L}\right)^k \!(f(x_0) - f^*) = O\!\left(\rho^k\right), \quad \rho = 1 - \frac{1}{\kappa}}$$

## 4.3 Backtracking Line Search (Armijo Rule)

When $L$ is unknown, we use backtracking: starting from $\alpha = \alpha_0$, shrink by factor $\beta \in (0,1)$ until the Armijo sufficient decrease condition holds:
$$f(x_k - \alpha \nabla f(x_k)) \leq f(x_k) - c\,\alpha\,\|\nabla f(x_k)\|^2$$
with $c \in (0, 0.5)$ (typically $c = 10^{-4}$).

In [ ]:
# =============================================================================
# Gradient Descent — fixed step and backtracking line search
# =============================================================================

def gradient_descent(func, x0, lr=LR_GD, max_iter=MAX_ITER, tol=TOL,
                     backtrack=False, alpha0=1.0, c=1e-4, beta=0.5):
    """Vanilla gradient descent with optional backtracking line search.

    Args:
        func:      callable (x) -> (f, g)
        x0:        initial point, shape (n,)
        lr:        fixed learning rate (used when backtrack=False)
        max_iter:  maximum number of iterations
        tol:       stop when ||g|| < tol
        backtrack: use Armijo backtracking if True
        alpha0:    initial step size for backtracking
        c:         Armijo sufficient decrease constant
        beta:      backtracking reduction factor

    Returns:
        traj:  list of iterates, each shape (n,)
        loss:  list of function values
    """
    x = x0.copy().astype(float)
    traj = [x.copy()]
    loss = []

    for _ in range(max_iter):
        f, g = func(x)
        loss.append(f)

        if np.linalg.norm(g) < tol:
            break

        if backtrack:
            alpha = alpha0
            while True:
                x_new = x - alpha * g
                f_new, _ = func(x_new)
                if f_new <= f - c * alpha * np.dot(g, g):
                    break
                alpha *= beta
                if alpha < 1e-16:
                    break
        else:
            alpha = lr

        x = x - alpha * g
        traj.append(x.copy())

    return traj, loss


# --- Demo on Rosenbrock ---
x0_rb = np.array([-1.2, 1.0])

traj_gd, loss_gd = gradient_descent(rosenbrock, x0_rb, lr=LR_GD)
traj_bt, loss_bt = gradient_descent(rosenbrock, x0_rb, backtrack=True, alpha0=1.0)

print(f"GD (fixed lr):     {len(loss_gd):4d} iters, final f = {loss_gd[-1]:.4e}")
print(f"GD (backtracking): {len(loss_bt):4d} iters, final f = {loss_bt[-1]:.4e}")

In [ ]:
# =============================================================================
# Convergence rate verification on a quadratic
# =============================================================================

def make_quadratic(kappa, n=2):
    """Build a quadratic with condition number kappa in n dimensions.

    Args:
        kappa: desired condition number L/mu
        n:     dimension

    Returns:
        func: callable (x) -> (f, g)
        Q:    PD matrix, shape (n, n)
        L:    largest eigenvalue
        mu:   smallest eigenvalue
    """
    eigs = np.linspace(1.0, kappa, n)
    Q    = np.diag(eigs)
    L    = eigs[-1]
    mu   = eigs[0]
    func = lambda x: quadratic(x, Q)
    return func, Q, L, mu


# Verify O(rho^k) convergence for strongly-convex quadratics
kappas = [1, 10, 100, 1000]
x0_q   = np.array([1.0, 1.0])  # f* = 0

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for kappa in kappas:
    func_q, Q, L, mu = make_quadratic(kappa, n=2)
    lr_q = 1.0 / L           # optimal fixed step = 1/L
    traj_q, loss_q = gradient_descent(func_q, x0_q, lr=lr_q, max_iter=500)
    rho = 1.0 - 1.0 / kappa
    iters = np.arange(len(loss_q))

    axes[0].semilogy(loss_q, label=f'$\\kappa={kappa}$')
    axes[1].semilogy(iters, rho**iters * loss_q[0], '--',
                     label=f'theory $(1-1/\\kappa)^k$, $\\kappa={kappa}$')
    axes[1].semilogy(loss_q, alpha=0.6)

axes[0].set_title('GD convergence — varying $\\kappa$')
axes[0].set_xlabel('Iteration'); axes[0].set_ylabel('$f(x_k) - f^*$')
axes[0].legend()

axes[1].set_title('GD vs theoretical rate $(1 - 1/\\kappa)^k$')
axes[1].set_xlabel('Iteration'); axes[1].set_ylabel('$f(x_k) - f^*$')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

# =============================================================================
# 5. Momentum Methods
# =============================================================================

## 5.1 Polyak's Heavy Ball

### Physics Analogy

Imagine a heavy ball rolling down a loss surface. Inertia carries it through flat regions and helps escape narrow valleys. The **heavy ball** method adds a momentum term:

$$\boxed{x_{k+1} = x_k - \alpha \nabla f(x_k) + \beta (x_k - x_{k-1})}$$

where $\beta \in [0, 1)$ is the **momentum coefficient** (or "friction" parameter).

Equivalently, introducing the velocity $v_k = x_k - x_{k-1}$:
$$v_{k+1} = \beta v_k - \alpha \nabla f(x_k), \qquad x_{k+1} = x_k + v_{k+1}.$$

For a quadratic $f$ with optimal parameters $\alpha^* = \left(\frac{2}{\sqrt{L}+\sqrt{\mu}}\right)^2$, $\beta^* = \left(\frac{\sqrt{L}-\sqrt{\mu}}{\sqrt{L}+\sqrt{\mu}}\right)^2$, heavy ball achieves:

$$\boxed{\|x_k - x^*\| \leq C \left(\frac{\sqrt{\kappa}-1}{\sqrt{\kappa}+1}\right)^k}$$

which is the same asymptotic rate as Nesterov on quadratics. However, heavy ball does **not** achieve accelerated rates on general convex functions.

## 5.2 Nesterov Accelerated Gradient

### Look-Ahead Intuition

Instead of computing the gradient at the current point $x_k$, Nesterov takes a "look-ahead" step to a predicted future point $y_k$, then evaluates the gradient there:

$$y_k = x_k + \frac{k-1}{k+2}(x_k - x_{k-1}) \quad (\text{momentum step})$$

$$\boxed{x_{k+1} = y_k - \frac{1}{L} \nabla f(y_k) \quad (\text{gradient step at look-ahead point})}$$

The momentum coefficient $\frac{k-1}{k+2}$ grows with $k$, providing stronger acceleration as iterations progress.

### Convergence Rate

For $L$-smooth convex $f$, Nesterov's method achieves the **optimal** first-order rate:

$$\boxed{f(x_k) - f^* \leq \frac{2L\,\|x_0 - x^*\|^2}{(k+1)^2} = O\!\left(\frac{1}{k^2}\right)}$$

This is provably optimal — no first-order method can do better in the worst case (Nesterov, 1983).

For strongly convex functions, using the **FISTA/restarting** variant achieves:
$$O\left(\left(1 - \frac{1}{\sqrt{\kappa}}\right)^k\right)$$

In [ ]:
# =============================================================================
# Polyak Heavy Ball
# =============================================================================

def heavy_ball(func, x0, lr=LR_MOMENTUM, beta=0.9, max_iter=MAX_ITER, tol=TOL):
    """Polyak heavy ball (momentum) gradient descent.

    Update rule:
        v_{k+1} = beta * v_k - lr * grad_f(x_k)
        x_{k+1} = x_k + v_{k+1}

    Args:
        func:     callable (x) -> (f, g)
        x0:       initial point, shape (n,)
        lr:       learning rate alpha
        beta:     momentum coefficient in [0, 1)
        max_iter: maximum iterations
        tol:      gradient norm stopping criterion

    Returns:
        traj: list of iterates, each shape (n,)
        loss: list of function values
    """
    x = x0.copy().astype(float)
    v = np.zeros_like(x)
    traj = [x.copy()]
    loss = []

    for _ in range(max_iter):
        f, g = func(x)
        loss.append(f)
        if np.linalg.norm(g) < tol:
            break
        v = beta * v - lr * g
        x = x + v
        traj.append(x.copy())

    return traj, loss


# =============================================================================
# Nesterov Accelerated Gradient
# =============================================================================

def nesterov_ag(func, x0, lr=LR_NESTEROV, max_iter=MAX_ITER, tol=TOL):
    """Nesterov accelerated gradient (FISTA-style momentum schedule).

    Update rule:
        y_k   = x_k + (k-1)/(k+2) * (x_k - x_{k-1})
        x_{k+1} = y_k - lr * grad_f(y_k)

    Args:
        func:     callable (x) -> (f, g)
        x0:       initial point, shape (n,)
        lr:       learning rate (should be <= 1/L)
        max_iter: maximum iterations
        tol:      gradient norm stopping criterion

    Returns:
        traj: list of iterates (x_k), each shape (n,)
        loss: list of function values at x_k
    """
    x      = x0.copy().astype(float)
    x_prev = x.copy()
    traj   = [x.copy()]
    loss   = []

    for k in range(1, max_iter + 1):
        f, g_x = func(x)
        loss.append(f)
        if np.linalg.norm(g_x) < tol:
            break

        # Look-ahead point
        mom   = (k - 1.0) / (k + 2.0)
        y     = x + mom * (x - x_prev)

        # Gradient step at look-ahead
        _, g_y = func(y)
        x_next = y - lr * g_y

        x_prev = x
        x      = x_next
        traj.append(x.copy())

    return traj, loss


# Demo
traj_hb, loss_hb = heavy_ball(rosenbrock, x0_rb, lr=LR_MOMENTUM, beta=0.9)
traj_na, loss_na = nesterov_ag(rosenbrock, x0_rb, lr=LR_NESTEROV)

print(f"Heavy Ball:        {len(loss_hb):4d} iters, final f = {loss_hb[-1]:.4e}")
print(f"Nesterov AG:       {len(loss_na):4d} iters, final f = {loss_na[-1]:.4e}")

In [ ]:
# =============================================================================
# Verify O(1/k^2) vs O(1/k) on a convex quadratic
# =============================================================================

func_q10, Q10, L10, mu10 = make_quadratic(kappa=10, n=10)
x0_q10 = rng.standard_normal(10)

lr_opt = 1.0 / L10
traj_gd_q,  loss_gd_q  = gradient_descent(func_q10, x0_q10, lr=lr_opt, max_iter=300)
traj_na_q,  loss_na_q  = nesterov_ag(func_q10, x0_q10, lr=lr_opt, max_iter=300)

iters = np.arange(1, min(len(loss_gd_q), len(loss_na_q)) + 1)
f0    = loss_gd_q[0]

fig, ax = plt.subplots(figsize=(10, 5))
ax.semilogy(iters, np.array(loss_gd_q[:len(iters)]), color=COLORS['gd'],
            label='GD (measured)')
ax.semilogy(iters, np.array(loss_na_q[:len(iters)]), color=COLORS['nesterov'],
            label='Nesterov (measured)')
# Theoretical reference curves
ax.semilogy(iters, f0 / iters, 'k--', alpha=0.6, label='$O(1/k)$ reference')
ax.semilogy(iters, f0 / iters**2, 'k:', alpha=0.6, label='$O(1/k^2)$ reference')
ax.set_xlabel('Iteration $k$')
ax.set_ylabel('$f(x_k) - f^*$')
ax.set_title('Convergence Rate Verification: GD vs Nesterov on Quadratic ($\\kappa=10$)')
ax.legend()
plt.tight_layout()
plt.show()

# =============================================================================
# 6. Adaptive Methods
# =============================================================================

## 6.1 Motivation — Per-Parameter Learning Rates

In problems with sparse gradients (e.g., NLP embeddings, one-hot features), most coordinates of $\nabla f$ are often zero. A single global learning rate is wasteful: parameters with rare but informative gradients need larger effective steps, while frequently-updated parameters benefit from smaller steps.

**Key idea:** scale the gradient coordinate-wise by the accumulated gradient information.

## 6.2 AdaGrad

Accumulate squared gradients $G_k = \sum_{t=1}^k g_t \odot g_t$ (element-wise), then:

$$\boxed{x_{k+1} = x_k - \frac{\alpha}{\sqrt{G_k + \epsilon}} \odot \nabla f(x_k)}$$

- Parameters with large historical gradients get small effective rates.
- Automatically adapts to rare features.
- **Problem:** $G_k$ grows monotonically — the effective rate $\to 0$ and learning can stop prematurely.

## 6.3 RMSProp

Fix AdaGrad's monotone accumulation with an **exponential moving average** (decay $\gamma \in (0,1)$):

$$v_k = \gamma\, v_{k-1} + (1-\gamma)\, g_k \odot g_k$$

$$\boxed{x_{k+1} = x_k - \frac{\alpha}{\sqrt{v_k + \epsilon}} \odot g_k}$$

RMSProp "forgets" old gradients, keeping the effective rate alive.

## 6.4 Adam — Adaptive Moment Estimation

Adam combines **momentum** (first moment) with **RMSProp** (second moment), plus **bias correction** to account for the zero-initialised moments:

$$m_k = \beta_1 m_{k-1} + (1-\beta_1) g_k \quad (\text{1st moment})$$
$$v_k = \beta_2 v_{k-1} + (1-\beta_2) g_k \odot g_k \quad (\text{2nd moment})$$
$$\hat{m}_k = \frac{m_k}{1 - \beta_1^k}, \quad \hat{v}_k = \frac{v_k}{1 - \beta_2^k} \quad (\text{bias correction})$$

$$\boxed{x_{k+1} = x_k - \frac{\alpha}{\sqrt{\hat{v}_k} + \epsilon}\, \hat{m}_k}$$

Defaults: $\beta_1 = 0.9$, $\beta_2 = 0.999$, $\epsilon = 10^{-8}$.

## 6.5 AMSGrad — Fixing Adam's Convergence

Adam can fail to converge on some convex problems (Reddi et al., 2018). AMSGrad fixes this by keeping a running maximum of the second moments:

$$\hat{v}_k^{\max} = \max(\hat{v}_{k-1}^{\max},\, \hat{v}_k)$$

$$\boxed{x_{k+1} = x_k - \frac{\alpha}{\sqrt{\hat{v}_k^{\max}} + \epsilon}\, \hat{m}_k}$$

This guarantees non-increasing effective learning rates, restoring convergence guarantees.

In [ ]:
# =============================================================================
# AdaGrad
# =============================================================================

def adagrad(func, x0, lr=LR_ADAGRAD, eps=1e-8, max_iter=MAX_ITER, tol=TOL):
    """AdaGrad optimizer (Duchi et al., 2011).

    Update rule:
        G_k   = G_{k-1} + g_k * g_k   (element-wise square accumulation)
        x_{k+1} = x_k - lr / sqrt(G_k + eps) * g_k

    Args:
        func:     callable (x) -> (f, g)
        x0:       initial point, shape (n,)
        lr:       global learning rate
        eps:      numerical stability constant
        max_iter: maximum iterations
        tol:      gradient norm stopping criterion

    Returns:
        traj: list of iterates, each shape (n,)
        loss: list of function values
    """
    x = x0.copy().astype(float)
    G = np.zeros_like(x)
    traj = [x.copy()]
    loss = []

    for _ in range(max_iter):
        f, g = func(x)
        loss.append(f)
        if np.linalg.norm(g) < tol:
            break
        G += g * g
        x  = x - lr / (np.sqrt(G) + eps) * g
        traj.append(x.copy())

    return traj, loss


# =============================================================================
# RMSProp
# =============================================================================

def rmsprop(func, x0, lr=LR_RMSPROP, gamma=0.9, eps=1e-8, max_iter=MAX_ITER, tol=TOL):
    """RMSProp optimizer (Hinton, 2012 — unpublished lecture notes).

    Update rule:
        v_k     = gamma * v_{k-1} + (1-gamma) * g_k^2
        x_{k+1} = x_k - lr / sqrt(v_k + eps) * g_k

    Args:
        func:     callable (x) -> (f, g)
        x0:       initial point, shape (n,)
        lr:       learning rate
        gamma:    decay factor for moving average
        eps:      numerical stability constant
        max_iter: maximum iterations
        tol:      gradient norm stopping criterion

    Returns:
        traj: list of iterates, each shape (n,)
        loss: list of function values
    """
    x = x0.copy().astype(float)
    v = np.zeros_like(x)
    traj = [x.copy()]
    loss = []

    for _ in range(max_iter):
        f, g = func(x)
        loss.append(f)
        if np.linalg.norm(g) < tol:
            break
        v = gamma * v + (1 - gamma) * g * g
        x = x - lr / (np.sqrt(v) + eps) * g
        traj.append(x.copy())

    return traj, loss


print("AdaGrad and RMSProp defined.")

In [ ]:
# =============================================================================
# Adam
# =============================================================================

def adam(func, x0, lr=LR_ADAM, beta1=0.9, beta2=0.999, eps=1e-8,
         max_iter=MAX_ITER, tol=TOL):
    """Adam optimizer (Kingma & Ba, 2015).

    Update rule:
        m_k     = beta1 * m_{k-1} + (1-beta1) * g_k
        v_k     = beta2 * v_{k-1} + (1-beta2) * g_k^2
        m_hat   = m_k / (1 - beta1^k)
        v_hat   = v_k / (1 - beta2^k)
        x_{k+1} = x_k - lr / (sqrt(v_hat) + eps) * m_hat

    Args:
        func:     callable (x) -> (f, g)
        x0:       initial point, shape (n,)
        lr:       learning rate (alpha)
        beta1:    first moment decay (default 0.9)
        beta2:    second moment decay (default 0.999)
        eps:      numerical stability constant (default 1e-8)
        max_iter: maximum iterations
        tol:      gradient norm stopping criterion

    Returns:
        traj: list of iterates, each shape (n,)
        loss: list of function values
    """
    x = x0.copy().astype(float)
    m = np.zeros_like(x)
    v = np.zeros_like(x)
    traj = [x.copy()]
    loss = []

    for k in range(1, max_iter + 1):
        f, g = func(x)
        loss.append(f)
        if np.linalg.norm(g) < tol:
            break
        m = beta1 * m + (1 - beta1) * g
        v = beta2 * v + (1 - beta2) * g * g
        m_hat = m / (1.0 - beta1**k)
        v_hat = v / (1.0 - beta2**k)
        x = x - lr / (np.sqrt(v_hat) + eps) * m_hat
        traj.append(x.copy())

    return traj, loss


# =============================================================================
# AMSGrad
# =============================================================================

def amsgrad(func, x0, lr=LR_AMSGRAD, beta1=0.9, beta2=0.999, eps=1e-8,
            max_iter=MAX_ITER, tol=TOL):
    """AMSGrad optimizer (Reddi et al., 2018).

    Differs from Adam by maintaining a running maximum of corrected
    second moments, ensuring non-increasing effective learning rates.

    Args:
        func:     callable (x) -> (f, g)
        x0:       initial point, shape (n,)
        lr:       learning rate
        beta1:    first moment decay
        beta2:    second moment decay
        eps:      numerical stability constant
        max_iter: maximum iterations
        tol:      gradient norm stopping criterion

    Returns:
        traj: list of iterates, each shape (n,)
        loss: list of function values
    """
    x     = x0.copy().astype(float)
    m     = np.zeros_like(x)
    v     = np.zeros_like(x)
    v_max = np.zeros_like(x)
    traj  = [x.copy()]
    loss  = []

    for k in range(1, max_iter + 1):
        f, g = func(x)
        loss.append(f)
        if np.linalg.norm(g) < tol:
            break
        m     = beta1 * m + (1 - beta1) * g
        v     = beta2 * v + (1 - beta2) * g * g
        m_hat = m / (1.0 - beta1**k)
        v_hat = v / (1.0 - beta2**k)
        v_max = np.maximum(v_max, v_hat)
        x     = x - lr / (np.sqrt(v_max) + eps) * m_hat
        traj.append(x.copy())

    return traj, loss


# Demo
traj_ad,  loss_ad  = adagrad(rosenbrock, x0_rb)
traj_rm,  loss_rm  = rmsprop(rosenbrock, x0_rb)
traj_am,  loss_am  = adam(rosenbrock, x0_rb)
traj_ams, loss_ams = amsgrad(rosenbrock, x0_rb)

print(f"AdaGrad:  {len(loss_ad):4d} iters, final f = {loss_ad[-1]:.4e}")
print(f"RMSProp:  {len(loss_rm):4d} iters, final f = {loss_rm[-1]:.4e}")
print(f"Adam:     {len(loss_am):4d} iters, final f = {loss_am[-1]:.4e}")
print(f"AMSGrad:  {len(loss_ams):4d} iters, final f = {loss_ams[-1]:.4e}")

# =============================================================================
# 7. Stochastic Gradient Descent
# =============================================================================

## 7.1 The Stochastic Setting

In machine learning the objective has the **finite-sum (empirical risk)** structure:

$$f(x) = \frac{1}{N} \sum_{i=1}^N f_i(x)$$

Computing the full gradient costs $O(N)$ per step. **Stochastic Gradient Descent (SGD)** replaces the full gradient with an unbiased estimator from a random mini-batch $\mathcal{B}_k \subseteq \{1,\ldots,N\}$ of size $B$:

$$g_k = \frac{1}{B}\sum_{i \in \mathcal{B}_k} \nabla f_i(x_k), \qquad \mathbb{E}[g_k \mid x_k] = \nabla f(x_k).$$

$$\boxed{x_{k+1} = x_k - \alpha_k\, g_k}$$

## 7.2 Convergence and Variance

The gradient estimator has variance $\sigma^2 = \mathbb{E}[\|g_k - \nabla f(x_k)\|^2]$. The SGD convergence bound becomes:

$$\mathbb{E}[f(x_k)] - f^* \leq \frac{\|x_0 - x^*\|^2}{2\alpha_k k} + \frac{\alpha_k \sigma^2}{2}$$

The second term is a **noise floor** — unlike deterministic GD, SGD cannot converge to zero error with a fixed step size.

## 7.3 Learning Rate Schedules

To achieve convergence, the step size must satisfy the **Robbins–Monro** conditions:
$$\sum_{k=1}^\infty \alpha_k = \infty, \qquad \sum_{k=1}^\infty \alpha_k^2 < \infty.$$

Common schedules:

| Schedule | $\alpha_k$ | Notes |
|---|---|---|
| **Constant** | $\alpha$ | Simple; noise floor persists |
| **$1/k$ decay** | $\alpha_0 / k$ | Satisfies Robbins–Monro; slow |
| **$1/\sqrt{k}$ decay** | $\alpha_0 / \sqrt{k}$ | Rate $O(1/\sqrt{k})$ for convex |
| **Cosine annealing** | $\alpha_{\min} + \frac{1}{2}(\alpha_{\max}-\alpha_{\min})(1+\cos(\pi k/T))$ | Popular in deep learning |
| **Warmup + decay** | Linear warmup, then cosine/step | Standard in Transformers |

In [ ]:
# =============================================================================
# Mini-batch SGD with variance reduction intuition
# We simulate the stochastic setting by adding Gaussian noise to the gradient.
# =============================================================================

def sgd_noisy(func, x0, lr=LR_SGD, noise_std=0.5, schedule='constant',
              max_iter=MAX_ITER, tol=TOL, seed=SEED):
    """SGD simulation with additive gradient noise (mimics mini-batch variance).

    Args:
        func:      callable (x) -> (f, g)
        x0:        initial point, shape (n,)
        lr:        base learning rate
        noise_std: standard deviation of gradient noise
        schedule:  'constant' | 'inv_sqrt' | 'inv_k' | 'cosine'
        max_iter:  maximum iterations
        tol:       gradient norm stopping criterion
        seed:      random seed for reproducibility

    Returns:
        traj: list of iterates, each shape (n,)
        loss: list of (true) function values
    """
    local_rng = np.random.default_rng(seed)
    x    = x0.copy().astype(float)
    traj = [x.copy()]
    loss = []

    for k in range(1, max_iter + 1):
        f, g = func(x)
        loss.append(f)
        if np.linalg.norm(g) < tol:
            break

        # Learning rate schedule
        if schedule == 'constant':
            alpha_k = lr
        elif schedule == 'inv_sqrt':
            alpha_k = lr / np.sqrt(k)
        elif schedule == 'inv_k':
            alpha_k = lr / k
        elif schedule == 'cosine':
            alpha_k = lr * 0.5 * (1 + np.cos(np.pi * k / max_iter))
        else:
            alpha_k = lr

        # Add stochastic noise to gradient
        g_noisy = g + noise_std * local_rng.standard_normal(x.shape)
        x = x - alpha_k * g_noisy
        traj.append(x.copy())

    return traj, loss


# Compare schedules on Rosenbrock
schedules = ['constant', 'inv_sqrt', 'inv_k', 'cosine']
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for sched in schedules:
    _, loss_s = sgd_noisy(rosenbrock, x0_rb, lr=0.002, noise_std=0.1,
                          schedule=sched, max_iter=1000)
    axes[0].semilogy(loss_s, label=sched, alpha=0.85)

axes[0].set_title('SGD: Learning Rate Schedules on Rosenbrock')
axes[0].set_xlabel('Iteration'); axes[0].set_ylabel('$f(x_k)$')
axes[0].legend()

# Noise level effect (constant schedule)
for std in [0.0, 0.05, 0.2, 0.5]:
    _, loss_n = sgd_noisy(rosenbrock, x0_rb, lr=0.002, noise_std=std,
                          schedule='constant', max_iter=800)
    axes[1].semilogy(loss_n, label=f'$\\sigma={std}$', alpha=0.85)

axes[1].set_title('SGD: Effect of Gradient Noise (constant lr)')
axes[1].set_xlabel('Iteration'); axes[1].set_ylabel('$f(x_k)$')
axes[1].legend()

plt.tight_layout()
plt.show()

# =============================================================================
# 8. Comparative Visualisation
# =============================================================================

We now run all methods on each test function and compare:
1. **Trajectory plots** — paths on 2D contour
2. **Convergence plots** — $f(x_k)$ vs iteration $k$
3. **Condition number study** — GD on quadratics with $\kappa \in \{1, 10, 100, 1000\}$

In [ ]:
# =============================================================================
# Unified runner — all optimisers on a given function
# =============================================================================

def run_all(func, x0, n_iter=1000):
    """Run all implemented optimisers on func from x0.

    Args:
        func:   callable (x) -> (f, g)
        x0:     initial point, shape (n,)
        n_iter: maximum iterations for each method

    Returns:
        results: dict mapping method name -> (traj, loss)
    """
    results = {}
    results['GD']       = gradient_descent(func, x0, lr=LR_GD,       max_iter=n_iter)
    results['GD-BT']    = gradient_descent(func, x0, backtrack=True,  max_iter=n_iter)
    results['HeavyBall']= heavy_ball(      func, x0, lr=LR_MOMENTUM,  max_iter=n_iter)
    results['Nesterov'] = nesterov_ag(     func, x0, lr=LR_NESTEROV,  max_iter=n_iter)
    results['AdaGrad']  = adagrad(         func, x0, lr=LR_ADAGRAD,   max_iter=n_iter)
    results['RMSProp']  = rmsprop(         func, x0, lr=LR_RMSPROP,   max_iter=n_iter)
    results['Adam']     = adam(            func, x0, lr=LR_ADAM,      max_iter=n_iter)
    results['AMSGrad']  = amsgrad(         func, x0, lr=LR_AMSGRAD,   max_iter=n_iter)
    return results


METHOD_COLORS = {
    'GD':        COLORS['gd'],
    'GD-BT':     '#17becf',
    'HeavyBall': COLORS['momentum'],
    'Nesterov':  COLORS['nesterov'],
    'AdaGrad':   COLORS['adagrad'],
    'RMSProp':   COLORS['rmsprop'],
    'Adam':      COLORS['adam'],
    'AMSGrad':   COLORS['amsgrad'],
}

print("Unified runner defined.")

In [ ]:
# =============================================================================
# Rosenbrock — trajectories on contour + convergence
# =============================================================================

x0_rb = np.array([-1.2, 1.0])
res_rb = run_all(rosenbrock, x0_rb, n_iter=2000)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# --- Contour plot with trajectories ---
ax = axes[0]
x1r = np.linspace(-2.0, 1.6, 500)
x2r = np.linspace(-0.5, 1.5, 500)
X1r, X2r = np.meshgrid(x1r, x2r)
Fr = 100*(X2r - X1r**2)**2 + (1 - X1r)**2
ax.contourf(X1r, X2r, np.log1p(Fr), levels=60, cmap='viridis', alpha=0.7)
ax.contour( X1r, X2r, np.log1p(Fr), levels=25, colors='white', alpha=0.25, linewidths=0.5)
ax.plot(1, 1, 'r*', markersize=16, zorder=10, label='$x^*$')

for name, (traj, _) in res_rb.items():
    pts = np.array(traj)
    ax.plot(pts[:, 0], pts[:, 1], '-', color=METHOD_COLORS[name],
            linewidth=1.2, alpha=0.85, label=name)
    ax.plot(pts[0, 0], pts[0, 1], 'o', color=METHOD_COLORS[name], markersize=5)

ax.set_xlim(-2.0, 1.6)
ax.set_ylim(-0.5, 1.5)
ax.set_title('Rosenbrock — Optimiser Trajectories', fontsize=13)
ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
ax.legend(fontsize=9, loc='upper left')

# --- Convergence curves ---
ax = axes[1]
for name, (_, loss) in res_rb.items():
    ax.semilogy(loss, color=METHOD_COLORS[name], label=name, alpha=0.9)

ax.set_title('Rosenbrock — Convergence (log scale)', fontsize=13)
ax.set_xlabel('Iteration $k$')
ax.set_ylabel('$f(x_k)$')
ax.legend(fontsize=9)
plt.suptitle('Rosenbrock Function: All Methods', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# Beale function — trajectories + convergence
# =============================================================================

x0_be = np.array([1.0, 1.5])
res_be = run_all(beale, x0_be, n_iter=2000)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

ax = axes[0]
x1b = np.linspace(-1.0, 4.5, 500)
x2b = np.linspace(-1.5, 2.0, 500)
X1b, X2b = np.meshgrid(x1b, x2b)
T1b = 1.5   - X1b + X1b*X2b
T2b = 2.25  - X1b + X1b*X2b**2
T3b = 2.625 - X1b + X1b*X2b**3
Fb  = T1b**2 + T2b**2 + T3b**2
ax.contourf(X1b, X2b, np.log1p(Fb), levels=60, cmap='plasma', alpha=0.7)
ax.contour( X1b, X2b, np.log1p(Fb), levels=25, colors='white', alpha=0.25, linewidths=0.5)
ax.plot(3, 0.5, 'r*', markersize=16, zorder=10, label='$x^*$')

for name, (traj, _) in res_be.items():
    pts = np.array(traj)
    # Clip for visibility
    mask = (pts[:,0] >= -1.0) & (pts[:,0] <= 4.5) & (pts[:,1] >= -1.5) & (pts[:,1] <= 2.0)
    pts_vis = pts[mask]
    if len(pts_vis) > 1:
        ax.plot(pts_vis[:,0], pts_vis[:,1], '-', color=METHOD_COLORS[name],
                linewidth=1.2, alpha=0.85, label=name)
        ax.plot(pts_vis[0,0], pts_vis[0,1], 'o', color=METHOD_COLORS[name], markersize=5)

ax.set_xlim(-1.0, 4.5)
ax.set_ylim(-1.5, 2.0)
ax.set_title('Beale — Optimiser Trajectories', fontsize=13)
ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
ax.legend(fontsize=9, loc='upper left')

ax = axes[1]
for name, (_, loss) in res_be.items():
    ax.semilogy(np.clip(loss, 1e-16, None), color=METHOD_COLORS[name], label=name, alpha=0.9)

ax.set_title('Beale — Convergence (log scale)', fontsize=13)
ax.set_xlabel('Iteration $k$')
ax.set_ylabel('$f(x_k)$')
ax.legend(fontsize=9)
plt.suptitle('Beale Function: All Methods', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# Condition number experiment — GD on quadratics
# =============================================================================

kappas = [1, 10, 100, 1000]
x0_q   = np.array([2.0, 1.0])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

cmap_kappa = plt.cm.cool
colors_kappa = [cmap_kappa(i / (len(kappas)-1)) for i in range(len(kappas))]

for i, kappa in enumerate(kappas):
    func_q, Q, L, mu = make_quadratic(kappa, n=2)
    lr_q = 1.0 / L
    _, loss_q = gradient_descent(func_q, x0_q, lr=lr_q, max_iter=1000)
    rho_theory = 1.0 - 1.0 / kappa
    iters = np.arange(len(loss_q))

    axes[0].semilogy(loss_q, color=colors_kappa[i], label=f'$\\kappa={kappa}$')
    # Plot theoretical rate
    axes[1].semilogy(iters, loss_q[0] * rho_theory**iters,
                     '--', color=colors_kappa[i], alpha=0.5)
    axes[1].semilogy(loss_q, color=colors_kappa[i], label=f'$\\kappa={kappa}$')

axes[0].set_title('GD on Quadratics — Measured Convergence', fontsize=13)
axes[0].set_xlabel('Iteration'); axes[0].set_ylabel('$f(x_k) - f^*$')
axes[0].legend()

axes[1].set_title('GD vs Theory $(1-1/\\kappa)^k$ (dashed)', fontsize=13)
axes[1].set_xlabel('Iteration'); axes[1].set_ylabel('$f(x_k) - f^*$')
axes[1].legend()

plt.suptitle('Condition Number Effect on Gradient Descent', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# Head-to-head convergence comparison — all methods on quadratic (kappa=50)
# =============================================================================

func_q50, Q50, L50, mu50 = make_quadratic(kappa=50, n=2)
x0_q50 = np.array([2.0, 1.0])

res_q50 = {}
lr_q50  = 1.0 / L50
res_q50['GD']        = gradient_descent(func_q50, x0_q50, lr=lr_q50,      max_iter=500)
res_q50['GD-BT']     = gradient_descent(func_q50, x0_q50, backtrack=True,  max_iter=500)
res_q50['HeavyBall'] = heavy_ball(      func_q50, x0_q50, lr=lr_q50*0.5,  max_iter=500)
res_q50['Nesterov']  = nesterov_ag(     func_q50, x0_q50, lr=lr_q50,      max_iter=500)
res_q50['AdaGrad']   = adagrad(         func_q50, x0_q50, lr=0.5,         max_iter=500)
res_q50['RMSProp']   = rmsprop(         func_q50, x0_q50, lr=0.05,        max_iter=500)
res_q50['Adam']      = adam(            func_q50, x0_q50, lr=0.05,        max_iter=500)
res_q50['AMSGrad']   = amsgrad(         func_q50, x0_q50, lr=0.05,        max_iter=500)

fig, ax = plt.subplots(figsize=(12, 6))
for name, (_, loss) in res_q50.items():
    ax.semilogy(np.clip(loss, 1e-16, None),
                color=METHOD_COLORS[name], label=name, alpha=0.9)

# Add theoretical GD rate
rho50 = 1 - 1.0 / 50
iters50 = np.arange(500)
ax.semilogy(res_q50['GD'][1][0] * rho50**iters50, 'k--', alpha=0.4, label='GD theory')

ax.set_title('All Methods on Quadratic ($\\kappa=50$)', fontsize=14)
ax.set_xlabel('Iteration $k$')
ax.set_ylabel('$f(x_k) - f^*$')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# Adaptive methods zoom-in: AdaGrad vs RMSProp — accumulated denominator effect
# =============================================================================

# Visualise how accumulated gradient magnitudes evolve (1D demo)
func_1d = lambda x: (0.5 * x[0]**2, np.array([x[0]]))
x0_1d   = np.array([10.0])

_, loss_ag_1d = adagrad( func_1d, x0_1d, lr=1.0, max_iter=200)
_, loss_rm_1d = rmsprop( func_1d, x0_1d, lr=0.5, max_iter=200)
_, loss_gd_1d = gradient_descent(func_1d, x0_1d, lr=0.1, max_iter=200)

fig, ax = plt.subplots(figsize=(10, 5))
ax.semilogy(loss_gd_1d, color=COLORS['gd'],       label='GD')
ax.semilogy(loss_ag_1d, color=COLORS['adagrad'],   label='AdaGrad')
ax.semilogy(loss_rm_1d, color=COLORS['rmsprop'],   label='RMSProp')
ax.set_title('AdaGrad vs RMSProp vs GD on a Simple Quadratic (1D)')
ax.set_xlabel('Iteration'); ax.set_ylabel('$f(x_k)$')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# Adam bias correction visualisation
# =============================================================================

# Show how bias-corrected vs raw moments differ in early iterations
beta1, beta2 = 0.9, 0.999
k_vals = np.arange(1, 101)

# Assume constant g = 1.0 at every step
# m_k = beta1^k * 0 + (1-beta1) * sum_{t=0}^{k-1} beta1^t  = 1 - beta1^k
# v_k = 1 - beta2^k
m_raw = 1.0 - beta1**k_vals
v_raw = 1.0 - beta2**k_vals
m_hat = m_raw / (1.0 - beta1**k_vals)   # = 1.0 everywhere
v_hat = v_raw / (1.0 - beta2**k_vals)   # = 1.0 everywhere

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(k_vals, m_raw, label='$m_k$ (raw)', color=COLORS['momentum'])
axes[0].axhline(1.0, color='k', linestyle='--', label='$\\hat{m}_k=1$ (corrected)')
axes[0].set_title('First Moment: Raw vs Bias-Corrected')
axes[0].set_xlabel('Iteration $k$'); axes[0].legend()

axes[1].plot(k_vals, v_raw, label='$v_k$ (raw)', color=COLORS['adam'])
axes[1].axhline(1.0, color='k', linestyle='--', label='$\\hat{v}_k=1$ (corrected)')
axes[1].set_title('Second Moment: Raw vs Bias-Corrected')
axes[1].set_xlabel('Iteration $k$'); axes[1].legend()

plt.suptitle('Adam Bias Correction: Why Early Iterations Need Rescaling',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# Comprehensive 3x2 comparison panel
# =============================================================================

fig, axes = plt.subplots(3, 2, figsize=(16, 18))

# ---- Row 0: Rosenbrock ----
ax_traj, ax_conv = axes[0]

x1r = np.linspace(-2.0, 1.6, 400)
x2r = np.linspace(-0.5, 1.5, 400)
X1r, X2r = np.meshgrid(x1r, x2r)
Fr  = 100*(X2r - X1r**2)**2 + (1 - X1r)**2
ax_traj.contourf(X1r, X2r, np.log1p(Fr), levels=50, cmap='viridis', alpha=0.65)
ax_traj.plot(1, 1, 'r*', markersize=14, zorder=10)

for name, (traj, loss) in res_rb.items():
    pts = np.array(traj)
    ax_traj.plot(pts[:,0], pts[:,1], '-', color=METHOD_COLORS[name],
                 linewidth=1.0, alpha=0.8, label=name)
    ax_conv.semilogy(loss, color=METHOD_COLORS[name], label=name, alpha=0.9)

ax_traj.set_xlim(-2.0, 1.6); ax_traj.set_ylim(-0.5, 1.5)
ax_traj.set_title('Rosenbrock — Trajectories'); ax_traj.legend(fontsize=7)
ax_conv.set_title('Rosenbrock — Convergence')
ax_conv.set_xlabel('Iteration'); ax_conv.set_ylabel('$f(x_k)$')
ax_conv.legend(fontsize=7)

# ---- Row 1: Beale ----
ax_traj, ax_conv = axes[1]

x1b = np.linspace(-0.5, 4.5, 400)
x2b = np.linspace(-1.0, 2.0, 400)
X1b, X2b = np.meshgrid(x1b, x2b)
T1b = 1.5 - X1b + X1b*X2b; T2b = 2.25 - X1b + X1b*X2b**2; T3b = 2.625 - X1b + X1b*X2b**3
Fb  = T1b**2 + T2b**2 + T3b**2
ax_traj.contourf(X1b, X2b, np.log1p(Fb), levels=50, cmap='plasma', alpha=0.65)
ax_traj.plot(3, 0.5, 'r*', markersize=14, zorder=10)

for name, (traj, loss) in res_be.items():
    pts = np.array(traj)
    vis = (pts[:,0]>=-0.5)&(pts[:,0]<=4.5)&(pts[:,1]>=-1.0)&(pts[:,1]<=2.0)
    if vis.sum() > 1:
        ax_traj.plot(pts[vis,0], pts[vis,1], '-', color=METHOD_COLORS[name],
                     linewidth=1.0, alpha=0.8, label=name)
    ax_conv.semilogy(np.clip(loss, 1e-16, None), color=METHOD_COLORS[name], label=name, alpha=0.9)

ax_traj.set_xlim(-0.5, 4.5); ax_traj.set_ylim(-1.0, 2.0)
ax_traj.set_title('Beale — Trajectories'); ax_traj.legend(fontsize=7)
ax_conv.set_title('Beale — Convergence')
ax_conv.set_xlabel('Iteration'); ax_conv.set_ylabel('$f(x_k)$')
ax_conv.legend(fontsize=7)

# ---- Row 2: Quadratic kappa=50 ----
ax_traj, ax_conv = axes[2]

x1q = np.linspace(-0.5, 2.5, 300)
x2q = np.linspace(-0.5, 1.5, 300)
X1q, X2q = np.meshgrid(x1q, x2q)
Fq_plot  = 0.5*(Q50[0,0]*X1q**2 + Q50[1,1]*X2q**2)
ax_traj.contourf(X1q, X2q, Fq_plot, levels=30, cmap='coolwarm', alpha=0.65)
ax_traj.plot(0, 0, 'r*', markersize=14, zorder=10)

for name, (traj, loss) in res_q50.items():
    pts = np.array(traj)
    vis = (pts[:,0]>=-0.5)&(pts[:,0]<=2.5)&(pts[:,1]>=-0.5)&(pts[:,1]<=1.5)
    if vis.sum() > 1:
        ax_traj.plot(pts[vis,0], pts[vis,1], '-', color=METHOD_COLORS[name],
                     linewidth=1.0, alpha=0.8, label=name)
    ax_conv.semilogy(np.clip(loss, 1e-16, None), color=METHOD_COLORS[name], label=name, alpha=0.9)

ax_traj.set_xlim(-0.5, 2.5); ax_traj.set_ylim(-0.5, 1.5)
ax_traj.set_title('Quadratic ($\\kappa=50$) — Trajectories'); ax_traj.legend(fontsize=7)
ax_conv.set_title('Quadratic ($\\kappa=50$) — Convergence')
ax_conv.set_xlabel('Iteration'); ax_conv.set_ylabel('$f(x_k) - f^*$')
ax_conv.legend(fontsize=7)

plt.suptitle('Comprehensive Comparison: All Optimisers on Three Test Functions',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# Final iterations reached and best function values summary
# =============================================================================

print("=" * 70)
print("ROSENBROCK RESULTS (from x0 = [-1.2, 1.0], 2000 iters)")
print(f"{'Method':<14} {'Iters':>7} {'Final f':>14} {'||x-x*||':>12}")
print("-" * 50)
x_star_rb = np.array([1.0, 1.0])
for name, (traj, loss) in res_rb.items():
    x_final = traj[-1]
    dist    = np.linalg.norm(x_final - x_star_rb)
    print(f"{name:<14} {len(loss):>7d} {loss[-1]:>14.4e} {dist:>12.4e}")

print()
print("=" * 70)
print("BEALE RESULTS (from x0 = [1.0, 1.5], 2000 iters)")
print(f"{'Method':<14} {'Iters':>7} {'Final f':>14} {'||x-x*||':>12}")
print("-" * 50)
x_star_be = np.array([3.0, 0.5])
for name, (traj, loss) in res_be.items():
    x_final = traj[-1]
    dist    = np.linalg.norm(x_final - x_star_be)
    print(f"{name:<14} {len(loss):>7d} {loss[-1]:>14.4e} {dist:>12.4e}")

# =============================================================================
# 9. Summary, Comparison Table & References
# =============================================================================

## 9.1 Method Comparison Table

| Method | Convergence Rate | Memory | Key Hyperparams | Notes |
|---|---|---|---|---|
| **GD (fixed)** | $O(1/k)$ convex; $O(\rho^k)$ s.c. | $O(n)$ | $\alpha$ | Simple; sensitive to $\alpha$ |
| **GD (backtrack)** | Same; adaptive $\alpha_k$ | $O(n)$ | $\alpha_0, c, \beta$ | More robust; extra func evals |
| **Heavy Ball** | $O(\rho_{\text{HB}}^k)$ on quadratics | $O(n)$ | $\alpha, \beta$ | No general convex guarantee |
| **Nesterov AG** | $O(1/k^2)$ convex; $O((1-1/\sqrt{\kappa})^k)$ s.c. | $O(n)$ | $\alpha$ | Optimal first-order rate |
| **AdaGrad** | $O(1/\sqrt{k})$ (sparse settings) | $O(n)$ | $\alpha$ | Monotone denominator $\to$ stalls |
| **RMSProp** | Heuristic; empirically fast | $O(n)$ | $\alpha, \gamma$ | EMA fix for AdaGrad |
| **Adam** | Non-convergent in worst case | $O(n)$ | $\alpha, \beta_1, \beta_2, \epsilon$ | De-facto DL standard |
| **AMSGrad** | Convergent (convex online) | $O(n)$ | $\alpha, \beta_1, \beta_2, \epsilon$ | Fixes Adam's convergence |
| **SGD (mini-batch)** | $O(1/\sqrt{k})$ with decay | $O(n)$ | $\alpha$, schedule | Generalises well in DL |

*s.c. = strongly convex, $\rho = 1 - 1/\kappa$, $\kappa = L/\mu$.*

## 9.2 Key Takeaways

1. **Condition number is the enemy.** GD convergence slows as $\kappa = L/\mu$ grows. On the Rosenbrock function ($\kappa \approx 2500$), convergence is painfully slow without acceleration.

2. **Nesterov acceleration is real.** The $O(1/k^2)$ rate vs $O(1/k)$ for GD on convex functions is tight and practically significant.

3. **Adaptive methods adapt to geometry.** AdaGrad/RMSProp/Adam implicitly precondition by the empirical Fisher information, which can be seen as approximating a diagonal Newton method.

4. **Adam ≠ convergent in general.** Reddi et al. (2018) showed a simple convex example where Adam diverges; AMSGrad restores convergence by using the running maximum of second moments.

5. **Stochastic methods need decaying step sizes for exact convergence.** With a fixed step size, SGD oscillates around the optimum; variance reduction methods (SVRG, SAG, SAGA) achieve linear convergence at the cost of storing gradients.

6. **In practice: Adam + warmup is the default.** For deep learning, Adam with learning rate warmup followed by cosine annealing dominates across vision, NLP, and RL benchmarks.

## 9.3 References

1. **Nocedal, J., & Wright, S. J.** (2006). *Numerical Optimization* (2nd ed.). Springer.
2. **Nesterov, Y.** (1983). A method for unconstrained convex minimization problem with the rate of convergence O(1/k²). *Doklady AN USSR*, 269, 543–547.
3. **Polyak, B. T.** (1964). Some methods of speeding up the convergence of iteration methods. *USSR Computational Mathematics and Mathematical Physics*, 4(5), 1–17.
4. **Duchi, J., Hazan, E., & Singer, Y.** (2011). Adaptive subgradient methods for online learning. *JMLR*, 12, 2121–2159.
5. **Kingma, D. P., & Ba, J.** (2015). Adam: A method for stochastic optimization. *ICLR 2015*.
6. **Reddi, S. J., Kale, S., & Kumar, S.** (2018). On the convergence of Adam and beyond. *ICLR 2018*.
7. **Bottou, L., Curtis, F. E., & Nocedal, J.** (2018). Optimization methods for large-scale machine learning. *SIAM Review*, 60(2), 223–311.
8. **Bubeck, S.** (2015). Convex optimization: Algorithms and complexity. *Foundations and Trends in Machine Learning*, 8(3–4), 231–357.